# 📋 [WEL07] 대량 부상자 발생 모의상황 의무지원 인계·기록 훈련
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/WEL07/WEL07_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 장병 복지·지원 (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.0시간) |
| **훈련 대상** | 대대 의무병, 응급구조사, 의무부사관, 지휘통제실 보고관 (3대 페르소나) |
| **핵심 역량** | MASCAL Triage, 의무기록 완결성, SBAR 지휘보고, 사후강평(AAR) |
| **실습 환경** | Google Colab, Jupyter Notebook, Markdown, JSON Dataset |

### 🎯 전체 산출물 디렉토리(서류철) 구조 및 저장 가이드

> **💡 디지털 작전 캐비닛 서류철(Binder) 개념:**  
> 사무실 작전 캐비닛에 [작전예규철], [훈련일지철], [결재보고철]을 따로 꽂아두듯, 본 과제의 산출물도 기능별로 서류철(Directory)을 나누어 보관합니다.  
> 학습자는 워크북의 `%%writefile docs/...` 코드 셀을 실행하기만 하면 자동으로 해당 서류철에 깔끔하게 보관되므로, 좌측 [파일 탐색기]에서 파일이 제자리에 잘 꽂혀 있는지만 확인하시면 됩니다.

| 디렉토리 | 군 실무 비유 명칭 | 주요 보관 산출물 | 학습자용 1줄 설명 |
| :--- | :--- | :--- | :--- |
| **`docs/`** | **기초 분석 및 기안철** | 상황 분석서(미션1), 교차검증표(미션5) | 훈련 시작 전 배경을 분석하고 데이터를 검증한 서류를 모아두는 곳 |
| **`sop/`** | **작전예규 및 체크리스트철** | 현장 점검표(미션2), 30초 휴대 포켓 가이드 | 현장 대원이 주머니에 넣고 바로 꺼내보는 행동 지침과 체크리스트 |
| **`scenarios/`** | **훈련 시나리오철** | MSEL 사건목록(미션3), TTX 각본 | 훈련 개시부터 종료까지 시간대별로 주입되는 가상 사건과 각본 |
| **`logs/`** | **훈련 일지 및 타임로그철** | Dry-run 기록지(미션4), 스트레스 로그(미션7) | 현장에서 스톱워치로 잰 실제 소요 시간과 관측 기록이 담긴 일지 |
| **`reports/`** | **지휘 보고철** | BLUF 1분 브리핑문, SBAR 인계서(미션6) | 의무대장이나 상급 제대에 올리는 공식 상황 보고서 |
| **`aar/`** | **사후강평 회의록철** | AAR 4단계 회의록(미션8), 5M 원인 분석표 | 훈련 종료 후 무엇이 잘되었고 무엇을 고쳐야 하는지 토의한 기록 |
| **`eval/`** | **성과 평가철** | 1차 vs 2차 전후 성과 비교표(미션9) | 개선된 SOP를 적용했을 때 시간과 오류가 얼마나 줄었는지 입증한 성적표 |
| **`package/`** | **완성 교범 및 배포철** | 국방 표준 TSSP 패키지, 교관 지침서(미션10) | 타 부대나 후임 부대에 그대로 전파할 수 있도록 묶은 최종 완성본 교범 |



In [ ]:
# [환경 준비] 산출물 디렉토리 일괄 생성 및 가상 데이터셋 무결성 확인
import os
import json

directories = [
    'docs',
    'sop',
    'scenarios',
    'logs',
    'reports',
    'aar',
    'eval',
    'package'
]

print('=== [1] 산출물 디렉토리 생성 ===')
for d in directories:
    os.makedirs(d, exist_ok=True)
    print(f'  📁 폴더 준비 완료: {d}/')

print('\n=== [2] 가상 데이터셋 무결성 확인 ===')
data_dir = '/home/subokim/Workspace/milicontent2/uploads/WEL07'
if not os.path.exists(data_dir):
    data_dir = '.'

dataset_count = 0
for i in range(1, 11):
    fname = f'wel07_m{i:02d}_dataset.json'
    fpath = os.path.join(data_dir, fname)
    if os.path.exists(fpath):
        with open(fpath, 'r', encoding='utf-8') as f:
            d = json.load(f)
            dataset_count += 1
            print(f'  ✅ 데이터셋 로드: {fname} (미션: {d.get("mission")})')
    else:
        print(f'  ⚠️ 데이터셋 탐색 중: {fname}')

print(f'\n총 {dataset_count}개 데이터셋 확인 완료. 실습 준비가 완료되었습니다!')


---

## 🎯 미션 01. 전시 대량 사상자(Mass Casualty, MASCAL) 발생 상황 시나리오 및 의무 체계 분석

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `docs/wel07_baseline_assessment.md`
- **AI 협업 프롬프트 질의 예시:**
> "가상 전훈 요약 카드 5건(L1~L5)과 모의환자 12명 상황표를 분석하여 '현장 처치', '분류', '후송' 3단계별 핵심 위험, 입력/산출 기록, 책임 부서를 표로 작성해줘. 또한 6개 직무 RACI 매트릭스와 15개 현장 체크 항목을 설계해줘."

- **AI 결과 검증 포인트 (HITL):**
> 상황 분석표 3단계 완비, 현장 체크 항목 15개 이상, 단일 책임(R) 원칙 준수 여부

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m01_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile docs/wel07_baseline_assessment.md
# [WEL07_M01] 미션 01 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m01_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 01 산출물 확인:** `docs/wel07_baseline_assessment.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 02. 중증도 분류(Triage) 기준표 및 의무기록 양식 설계

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `sop/wel07_field_checklist.md`
- **AI 협업 프롬프트 질의 예시:**
> "긴급(Red), 응급(Yellow), 비응급(Green), 지연(Black) 4색 기준표를 기반으로, 야전에서 30초 내 식별 가능한 기록 규칙과 7대 필수 기재란이 포함된 의무기록 양식, 그리고 30초 퀵 포켓 가이드를 설계해줘."

- **AI 결과 검증 포인트 (HITL):**
> 의학적 단정 표현 배제, 7대 필수 기재란 명시, 포켓 가이드 30초 가독성 충족

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m02_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile sop/wel07_field_checklist.md
# [WEL07_M02] 미션 02 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m02_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 02 산출물 확인:** `sop/wel07_field_checklist.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 03. 현실적 대량전상자 시뮬레이션 시나리오 및 MSEL(주요사건목록) 설계

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `scenarios/wel07_msel_scenario.md`
- **AI 협업 프롬프트 질의 예시:**
> "현장 처치반-분류소-후송반 3개 구간의 인계 프로세스를 정의하고, 10건의 MSEL(주요사건목록)을 정상 6 : 돌발 난제 4 비율로 설계해줘. 식별표 분실, 통신 두절 등 돌발 변수와 도상 연습(TTX) 점검표를 포함해줘."

- **AI 결과 검증 포인트 (HITL):**
> 정상 6 : 비정상 4 비율 준수, 타임라인(+00:00~+08:00) 일치, 안전 통제 수칙 포함

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m03_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile scenarios/wel07_msel_scenario.md
# [WEL07_M03] 미션 03 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m03_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 03 산출물 확인:** `scenarios/wel07_msel_scenario.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 04. 대량전상자 처치 및 환자인계 실전 모의훈련(Dry-run) 수행

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `logs/wel07_dryrun_execution_log.md`
- **AI 협업 프롬프트 질의 예시:**
> "모의환자 8명(M-01~M-08)에 대한 1차 Dry-run 수행 결과를 바탕으로 분류 일람표, 관측 기록지, 사상자 집결지(CCP) 병목 구간 기록표를 작성하고, 3대 성과 지표(보고 지연시간, SOP 준수율, 누락률)를 산출해줘."

- **AI 결과 검증 포인트 (HITL):**
> 보고 지연 3분 이내 기준 대조, 56칸 필수 기재란 누락률 계측, 병목 4대 구간 대책 제시

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m04_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile logs/wel07_dryrun_execution_log.md
# [WEL07_M04] 미션 04 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m04_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 04 산출물 확인:** `logs/wel07_dryrun_execution_log.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 05. 다출처 의무 데이터 수집 및 정합성 교차검증 체계 구축

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `docs/wel07_cross_verification_sheet.md`
- **AI 협업 프롬프트 질의 예시:**
> "수기 TCCC 카드, 분류 일람표, 후송 순서표, 관측 기록 등 4개 출처에서 수집된 12개 대조 항목을 비교하여 불일치 오류를 식별하고, 오류 발생 시 3단계 보고 경로 및 인간 검토(HITL) 서명 양식을 작성해줘."

- **AI 결과 검증 포인트 (HITL):**
> 식별표 분실(TMP-01) 및 중증도 전도 오류 식별, 3단계 보고 경로 명문화, HITL 책임분계

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m05_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile docs/wel07_cross_verification_sheet.md
# [WEL07_M05] 미션 05 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m05_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 05 산출물 확인:** `docs/wel07_cross_verification_sheet.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 06. 환자인계 시 의무 지휘보고 및 상황공유 프로토콜 개발

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `reports/wel07_handover_protocol.md`
- **AI 협업 프롬프트 질의 예시:**
> "의무대장 앞 1분 이내 구두 상황 보고문과 SBAR(상황-배경-평가-제안) 구조화 보고서를 작성해줘. 또한 후송 인계 품질 체크리스트 및 인수자의 5가지 돌발 질문(Q1~Q5)에 대한 즉각 대응 스크립트를 작성해줘."

- **AI 결과 검증 포인트 (HITL):**
> BLUF 원칙 서두 배치, SBAR 4단계 충족, 인계자-인수자 상호 서명 및 돌발 질의 답변 완비

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m06_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile reports/wel07_handover_protocol.md
# [WEL07_M06] 미션 06 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m06_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 06 산출물 확인:** `reports/wel07_handover_protocol.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 07. 극한 스트레스 및 자원 결핍 상황 대응 훈련 실시

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `logs/wel07_stress_drill_results.md`
- **AI 협업 프롬프트 질의 예시:**
> "의무 물자 고갈, 통신 두절, 지휘관 유고 등 6대 스트레스 사건 카드 주입에 따른 대응 기록표와 비상 직무 위임 순서표(Fail-safe)를 작성해줘. 전환 시간(2분 이내)과 안전 중지(Safety Pause) 프로토콜을 포함해줘."

- **AI 결과 검증 포인트 (HITL):**
> 스트레스 사건 6종 대응 수칙, 임무형 지휘 대행 서열 4단계, 2분 내 전환 타임로그 확인

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m07_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile logs/wel07_stress_drill_results.md
# [WEL07_M07] 미션 07 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m07_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 07 산출물 확인:** `logs/wel07_stress_drill_results.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 08. 사후강평(AAR)을 통한 표준절차(SOP) 개선사항 도출

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `aar/wel07_aar_session_minutes.md`
- **AI 협업 프롬프트 질의 예시:**
> "훈련 로그를 바탕으로 AAR 4단계(목표-실제-원인-개선) 회의록을 작성하고, 인계 누락과 분류 지연에 대한 5M(절차·양식·환경·숙달·통신) 근본 원인 분석표와 SOP 3단 비교표(현행-개정안-사유)를 작성해줘."

- **AI 결과 검증 포인트 (HITL):**
> 비난 없는 사실 중심 분석, 5M 전 영역 결함 도출, 3단 비교표 상 구체적 행동 지침화

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m08_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile aar/wel07_aar_session_minutes.md
# [WEL07_M08] 미션 08 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m08_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 08 산출물 확인:** `aar/wel07_aar_session_minutes.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 09. 개선된 표준절차(SOP) 기반 2차 훈련 및 효과성 검증

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `eval/wel07_rerun_performance_comparison.md`
- **AI 협업 프롬프트 질의 예시:**
> "개정된 SOP를 적용한 2차 훈련 결과를 바탕으로 1차 대비 성과 비교표(시간 30% 이상 단축, 누락 0건)를 작성하고, 야간·악천후 환경 변수 적응성 평가 및 분기별 보수 교육(Refresher Training) 방안을 수립해줘."

- **AI 결과 검증 포인트 (HITL):**
> 지표 단축률(시간 35% 단축 등) 정량 명시, 극저조도/방수 대책 검증, 분기 주기 명시

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m09_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile eval/wel07_rerun_performance_comparison.md
# [WEL07_M09] 미션 09 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m09_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 09 산출물 확인:** `eval/wel07_rerun_performance_comparison.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


---

## 🎯 미션 10. 대량전상자 의무지원 및 환자인계 표준 교육훈련 패키지 제작

### 1. 미션 목표 및 핵심 산출물
- **필수 산출물 경로:** `package/wel07_standard_training_package.md`
- **AI 협업 프롬프트 질의 예시:**
> "국방 표준 교육훈련 패키지(TSSP) 5개 장(개요-SOP-MSEL-체크리스트-교관지침서) 편철 목차와 본문을 작성해줘. 5단계 관찰 평가 루브릭, 타 부대 전파성 검토 의견표, 군의관 감수 확인서를 포함해줘."

- **AI 결과 검증 포인트 (HITL):**
> 5개 장 완비, 5단계 정량 평가 루브릭, 타 부대 이식성 검수표, 군의관 공식 승인 서명란

### 2. 가상 데이터셋 확인 및 파일 생성
아래 코드 셀을 실행하여 가상 데이터셋(`wel07_m10_dataset.json`)을 기반으로 산출물을 생성하십시오.


In [ ]:
%%writefile package/wel07_standard_training_package.md
# [WEL07_M10] 미션 10 실습 산출물

## 1. 개요 및 목적
<!-- [실습 과업] 가이드스텝과 AI 코파일럿 질의를 참고하여 본 미션의 개요와 실습 목적을 기술하십시오. -->

## 2. 핵심 분석 및 설계 매트릭스
<!-- [실습 과업] 제공된 가상 데이터셋(wel07_m10_dataset.json)을 로드하여 핵심 테이블을 완성하십시오. -->

## 3. 현장 표준 점검표 및 세부 실행 계획
<!-- [실습 과업] 단정적 표현을 배제하고 군사적·의학적 무결성을 검증한 체크리스트 및 절차를 기술하십시오. -->

## 4. 검증 결과 및 책임자 서명
<!-- [실습 과업] PASS 기준 충족 여부를 점검하고 검토자 서명을 완료하십시오. -->



**✅ 미션 10 산출물 확인:** `package/wel07_standard_training_package.md` 파일이 성공적으로 작성되었습니다. 파일 탐색기에서 생성 결과를 확인하십시오.


In [ ]:
# [최종 검증] 10개 미션 필수 산출물 무결성 자동 채점 및 패키징
import os
import zipfile

expected_files = [
    'docs/wel07_baseline_assessment.md',
    'sop/wel07_field_checklist.md',
    'scenarios/wel07_msel_scenario.md',
    'logs/wel07_dryrun_execution_log.md',
    'docs/wel07_cross_verification_sheet.md',
    'reports/wel07_handover_protocol.md',
    'logs/wel07_stress_drill_results.md',
    'aar/wel07_aar_session_minutes.md',
    'eval/wel07_rerun_performance_comparison.md',
    'package/wel07_standard_training_package.md'
]

print('=== 📋 WEL07 최종 산출물 무결성 판정 ===')
pass_count = 0
for idx, fpath in enumerate(expected_files, 1):
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        if size > 100:
            print(f'  [미션 {idx:02d}] PASS: {fpath} ({size:,} bytes)')
            pass_count += 1
        else:
            print(f'  [미션 {idx:02d}] WARNING: {fpath} 내용 부족 ({size} bytes)')
    else:
        print(f'  [미션 {idx:02d}] FAIL: {fpath} 파일 누락')

print('-' * 50)
score = (pass_count / len(expected_files)) * 100
print(f'최종 채점 결과: {pass_count}/{len(expected_files)} 통과 ({score:.1f}점)')

if pass_count == len(expected_files):
    os.makedirs('package', exist_ok=True)
    zip_path = 'package/WEL07_submission_package.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for f in expected_files:
            zipf.write(f)
    print('🎉 축하합니다! WEL07 전 미션 필수 평가 기준을 완벽히 PASS하였습니다!')
    print(f'📦 최종 제출 패키지가 생성되었습니다: {zip_path}')
else:
    print('⚠️ 누락되거나 내용이 부족한 산출물을 확인하고 해당 코드 셀을 재실행하십시오.')
